# 03 · The unfolded net used for indexing

Every vertex has an integer grid position `(vertex.x, vertex.y)`. Its neighbours on the sphere are always
one of the six grid offsets `(+1, 0) (-1, 0) (0, +1) (0, -1) (+1, +1) (-1, -1)`, which is what makes
neighbour search fast. Vertices on the boundary of the net are stored several times
(`vertex.same_vertices`) so the net can be "glued" back into a sphere.

Script version: [`examples/03_unfolded_net.py`](../03_unfolded_net.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from dome_utils import unique_mesh
from IPython.display import display
from matplotlib.collections import PolyCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome

# Shear the index grid so that every triangle becomes equilateral:
#   (x, y) -> x * (1, 0) + y * (-1/2, sqrt(3)/2)
SHEAR = np.array([[1.0, 0.0], [-0.5, np.sqrt(3) / 2]])

## Classify the vertices

The 12 icosahedron corners have 5 neighbours on the sphere; other seam vertices are stored twice;
everything else is an interior vertex stored once.

In [ ]:
def net_layout(freq):
    dome = GeodesicDome(freq)
    vertices = dome.get_all_vertices()
    triangles = dome.get_all_triangles().reshape(-1, 3)   # also assigns vertex.id
    grid = np.array([[v.x, v.y] for v in vertices], dtype=float)

    points, faces, index_map = unique_mesh(dome)
    degree = np.zeros(len(points), dtype=int)
    edges = {tuple(sorted(e)) for f in faces for e in ((f[0], f[1]), (f[1], f[2]), (f[2], f[0]))}
    for a, b in edges:
        degree[a] += 1
        degree[b] += 1
    is_corner = degree[index_map] == 5
    is_seam = np.array([bool(v.same_vertices) for v in vertices]) & ~is_corner
    return dict(dome=dome, vertices=vertices, triangles=triangles, grid=grid, net=grid @ SHEAR,
                points=points, is_corner=is_corner, is_seam=is_seam, is_inner=~is_corner & ~is_seam)


def draw_net(ax, L, coords, title):
    ax.add_collection(PolyCollection(coords[L['triangles']], facecolors='#e8eef6',
                                     edgecolors='#8aa4c8', linewidths=0.6))
    ax.scatter(*coords[L['is_inner']].T, s=14, c='#4a6fa5', label='interior vertex', zorder=3)
    ax.scatter(*coords[L['is_seam']].T, s=26, c='#e6892e', label='seam vertex (stored 2x)', zorder=3)
    ax.scatter(*coords[L['is_corner']].T, s=60, c='#c0392b', marker='p',
               label='icosahedron corner (5 neighbours)', zorder=4)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.autoscale()


FREQ = 3
L = net_layout(FREQ)
print(f"GeodesicDome(frequency={FREQ}): {len(L['vertices'])} stored vertices, "
      f"{len(L['points'])} unique points, {len(L['triangles'])} triangles")
print(f"interior {L['is_inner'].sum()}, seam {L['is_seam'].sum()}, corner copies {L['is_corner'].sum()}")

## The index grid and the icosahedron net

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1, 1.25]})
draw_net(axes[0], L, L['grid'], 'index grid  (vertex.x, vertex.y)')
draw_net(axes[1], L, L['net'], 'same grid, sheared: the icosahedron net')

# the six grid neighbours of one vertex
v = L['dome'].get_vertex_at(4, 4)
for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (-1, -1)):
    axes[0].annotate('', xy=(v.x + 0.8 * dx, v.y + 0.8 * dy), xytext=(v.x, v.y),
                     arrowprops=dict(arrowstyle='->', color='black', lw=1.2))
axes[0].text(v.x + 0.15, v.y - 0.45, f'({v.x},{v.y})', fontsize=9)
axes[0].legend(loc='upper left', fontsize=8)
fig.tight_layout()
plt.show()

### Try it: follow a vertex across the seams

Pick a frequency and a stored vertex. The selected vertex is circled, its **seam copies** are drawn as
squares and its **neighbours on the sphere** (`get_neighbours`) as green diamonds. For a seam vertex the
neighbours sit on both sides of the cut, which is how a ring continues across the edges of the net.

In [ ]:
freq = widgets.IntSlider(value=3, min=1, max=8, description='frequency')
index = widgets.IntSlider(value=0, min=0, max=len(L['vertices']) - 1, description='vertex')
sheared = widgets.Checkbox(value=True, description='sheared net')
cache = {}


def layout(f):
    if f not in cache:
        cache[f] = net_layout(f)
    return cache[f]


def on_freq(change):
    index.max = len(layout(change['new'])['vertices']) - 1


freq.observe(on_freq, names='value')
live = LiveFigure(figsize=(10, 6))


def draw(freq, index, sheared):
    L = layout(freq)
    coords = L['net'] if sheared else L['grid']
    v = L['vertices'][min(index, len(L['vertices']) - 1)]
    dome = L['dome']
    dome.unmark_vertices()
    neighbours = dome.get_neighbours(v, False)
    dome.unmark_vertices()

    to_xy = (lambda x, y: np.array([x, y], float) @ SHEAR) if sheared else (lambda x, y: np.array([x, y], float))
    fig = live.clear()
    ax = fig.add_subplot()
    draw_net(ax, L, coords, f'GeodesicDome({freq}): vertex ({v.x}, {v.y}), {len(neighbours)} neighbours')
    if neighbours:
        ax.scatter(*np.array([to_xy(n.x, n.y) for n in neighbours]).T, s=110, marker='D',
                   facecolors='none', edgecolors='#27ae60', linewidths=2, zorder=5, label='neighbours')
    copies = v.same_vertices or []
    if copies:
        ax.scatter(*np.array([to_xy(s.x, s.y) for s in copies]).T, s=180, marker='s',
                   facecolors='none', edgecolors='#8e44ad', linewidths=2, zorder=5, label='seam copies')
    ax.scatter(*to_xy(v.x, v.y), s=260, facecolors='none', edgecolors='black', linewidths=2.5, zorder=6,
               label='selected vertex')
    ax.legend(loc='upper left', fontsize=8)
    fig.tight_layout()
    live.refresh()


out = widgets.interactive_output(draw, dict(freq=freq, index=index, sheared=sheared))
display(widgets.HBox([freq, index, sheared]), out)
live.show()